<a href="https://colab.research.google.com/github/srcorreia3001/sistema_de_recomendacao_de_produtos/blob/main/Sistema_de_Recomenda%C3%A7%C3%A3o_de_Produtos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# para evitar poluição na tela devido a um comportamento interno do jupyter notebook rodando em versões mais recentes  do Python
import warnings
warnings.filterwarnings('ignore', category=DeprecationWarning)

In [ ]:
import pandas as pd
import numpy as np

# 1. Simulando um banco de dados de transações
np.random.seed(42)

n_transacoes = 5000

usuarios_ids = [f"User_{np.random.randint(1, 200):03d}" for _ in range(n_transacoes)]
produtos_catalogo = {
    "Smartphone": 1500.0,
    "Capinha de Celular": 50.0,
    "Película de Vidro": 30.0,
    "Fone Bluetooth": 200.0,
    "Smartwatch": 400.0,
    "Notebook": 3500.0,
    "Mouse Gamer": 120.0,
    "Teclado Mecânico": 250.0,
    "Mochila para Notebook": 180.0
}

# Criando dados onde certos produtos costumam ser comprados juntos
dados = []
for i in range(n_transacoes):
    user = np.random.choice(usuarios_ids)
    # Escolhe um produto principal
    produto_principal = np.random.choice(list(produtos_catalogo.keys()))
    preco = produtos_catalogo[produto_principal]

    # Simula cross-selling natural (ex: quem compra smartphone tem chance de levar capinha)
    itens_pedido = [produto_principal]
    if produto_principal == "Smartphone" and np.random.rand() > 0.3:
        itens_pedido.append("Capinha de Celular")
        itens_pedido.append("Película de Vidro")
    elif produto_principal == "Notebook" and np.random.rand() > 0.4:
        itens_pedido.append("Mouse Gamer")
        itens_pedido.append("Mochila para Notebook")

    for item in set(itens_pedido):
        dados.append({
            "id_transacao": f"T_{i+1:04d}",
            "id_usuario": user,
            "produto": item,
            "preco": produtos_catalogo[item]
        })

df_transacoes = pd.DataFrame(dados)

print("--- Amostra dos Dados Fictícios de Transações ---")
print(df_transacoes.head(10))

# 2. Análise Exploratória Básica (Respondendo a perguntas iniciais)
print("\n--- Produtos mais vendidos ---")
print(df_transacoes['produto'].value_counts())

print("\n--- Ticket Médio Geral por Transação ---")
ticket_transacao = df_transacoes.groupby('id_transacao')['preco'].sum()
print(f"R$ {ticket_transacao.mean():.2f}")

--- Amostra dos Dados Fictícios de Transações ---
  id_transacao id_usuario             produto   preco
0       T_0001   User_012         Mouse Gamer   120.0
1       T_0002   User_138          Smartwatch   400.0
2       T_0003   User_086  Capinha de Celular    50.0
3       T_0003   User_086   Película de Vidro    30.0
4       T_0003   User_086          Smartphone  1500.0
5       T_0004   User_168  Capinha de Celular    50.0
6       T_0004   User_168   Película de Vidro    30.0
7       T_0004   User_168          Smartphone  1500.0
8       T_0005   User_001            Notebook  3500.0
9       T_0005   User_001         Mouse Gamer   120.0

--- Produtos mais vendidos ---
produto
Capinha de Celular       945
Película de Vidro        930
Mouse Gamer              913
Mochila para Notebook    883
Notebook                 569
Smartphone               561
Smartwatch               557
Teclado Mecânico         548
Fone Bluetooth           522
Name: count, dtype: int64

--- Ticket Médio Geral por T

In [ ]:
# Exibindo as regras de forma humanizada e legível
print("--- Principais Regras de Cross-selling Encontradas ---\n")
for idx, row in rules.head(5).iterrows():
    antecedente = list(row['antecedents'])
    consequente = list(row['consequents'])
    confianca = row['confidence'] * 100
    lift = row['lift']
    print(f"👉 Quem compra: {antecedente}")
    print(f"   🔹 Leva também: {consequente}")
    print(f"   📊 Confiança: {confianca:.1f}% | Lift: {lift:.2f}\n")

--- Principais Regras de Cross-selling Encontradas ---

👉 Quem compra: ['Capinha de Celular', 'Película de Vidro']
   🔹 Leva também: [np.str_('Smartphone')]
   📊 Confiança: 100.0% | Lift: 8.91

👉 Quem compra: [np.str_('Mouse Gamer'), 'Mochila para Notebook']
   🔹 Leva também: [np.str_('Notebook')]
   📊 Confiança: 100.0% | Lift: 8.79

👉 Quem compra: [np.str_('Notebook'), np.str_('Mouse Gamer')]
   🔹 Leva também: ['Mochila para Notebook']
   📊 Confiança: 100.0% | Lift: 5.66

👉 Quem compra: [np.str_('Notebook'), 'Mochila para Notebook']
   🔹 Leva também: [np.str_('Mouse Gamer')]
   📊 Confiança: 100.0% | Lift: 5.48

👉 Quem compra: ['Capinha de Celular', np.str_('Smartphone')]
   🔹 Leva também: ['Película de Vidro']
   📊 Confiança: 100.0% | Lift: 5.38



In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

# 1. Criando a matriz Usuário x Produto (User-Item Matrix)
# Indicando se o usuário já comprou o produto (1) ou não (0)
user_item_matrix = pd.crosstab(df_transacoes['id_usuario'], df_transacoes['produto'])

# 2. Calculando a Similaridade de Cosseno entre os Produtos (Item-based Collaborative Filtering)
# Transpomos a matriz para que as linhas sejam os produtos
item_similarity = cosine_similarity(user_item_matrix.T)
df_item_similarity = pd.DataFrame(item_similarity,
                                  index=user_item_matrix.columns,
                                  columns=user_item_matrix.columns)

print("--- Matriz de Similaridade entre Produtos (Amostra) ---")
print(df_item_similarity.head(4).iloc[:, :4])

# 3. Função de Recomendação Personalizada para um Usuário
def recomendar_para_usuario(id_usuario, n_recomendacoes=3):
    if id_usuario not in user_item_matrix.index:
        return f"Usuário {id_usuario} não encontrado."

    # Produtos que o usuário já comprou
    compras_usuario = user_item_matrix.loc[id_usuario]
    produtos_comprados = compras_usuario[compras_usuario > 0].index.tolist()

    # Pontuação de recomendação baseada na similaridade dos produtos comprados
    score_recomendacao = pd.Series(0.0, index=df_item_similarity.columns)
    for prod in produtos_comprados:
        score_recomendacao += df_item_similarity[prod]

    # Remove os produtos que o usuário já comprou da lista de sugestões
    score_recomendacao = score_recomendacao.drop(produtos_comprados)

    # Retorna os top N produtos recomendados
    recomendados = score_recomendacao.sort_values(ascending=False).head(n_recomendacoes).index.tolist()
    return produtos_comprados, recomendados

# Testando a função para um usuário aleatório do dataset
usuario_teste = user_item_matrix.index[10]
comprados, sugeridos = recomendar_para_usuario(usuario_teste)

print(f"\n--- Testando Recomendações para o {usuario_teste} ---")
print(f"🛒 Produtos já comprados pelo usuário: {comprados}")
print(f"💡 Produtos recomendados (Personalizados): {sugeridos}")

--- Matriz de Similaridade entre Produtos (Amostra) ---
produto                Capinha de Celular  Fone Bluetooth  \
produto                                                     
Capinha de Celular               1.000000        0.788223   
Fone Bluetooth                   0.788223        1.000000   
Mochila para Notebook            0.833273        0.748909   
Mouse Gamer                      0.822969        0.744584   

produto                Mochila para Notebook  Mouse Gamer  
produto                                                    
Capinha de Celular                  0.833273     0.822969  
Fone Bluetooth                      0.748909     0.744584  
Mochila para Notebook               1.000000     0.889602  
Mouse Gamer                         0.889602     1.000000  

--- Testando Recomendações para o User_011 ---
🛒 Produtos já comprados pelo usuário: ['Capinha de Celular', np.str_('Fone Bluetooth'), 'Mochila para Notebook', np.str_('Mouse Gamer'), np.str_('Notebook'), 'Película d